# 第16次课：GeoPandas 基础（模块四开篇）

**地学编程基础** | 2学时（90分钟）

---

## 🎉 模块四正式开始 —— 真正的 GIS 旅程

前 15 节课你掌握了：
- ✅ Python 编程基础（模块一，8 课）
- ✅ 函数+文件+异常（模块二，4 课）
- ✅ 数据分析（模块三，3 课）

**今天起 —— 终于进入地理信息系统的世界**！

| 课次 | 主题 | 学到什么 |
|---|---|---|
| **第 16 次（今天）** | **GeoPandas 基础** | **读 Shapefile、画地图、几何属性** |
| 第 17 次 | GeoPandas 空间分析 | 缓冲区、空间查询、空间连接 |
| 第 18 次 | Rasterio 栅格遥感 | 读 GeoTIFF、NDVI 实战 |
| 第 19 次 | 可视化与综合应用 | matplotlib 高级、地图美化 |
| 第 20 次 | 综合项目（期末） | 完整地学分析 |

---

## 📚 本节课学习目标

完成本节课后，你应该能够：

1. **理解** GIS 矢量数据的两个核心：**几何**（geometry）+ **属性**（attributes）；
2. **理解** CRS（坐标参考系）的基本概念；
3. **认识** Shapefile 的文件结构（.shp/.shx/.dbf/.prj 必须齐全）；
4. **掌握** `gpd.read_file()` 读取矢量数据；
5. **掌握** GeoDataFrame 的特殊属性（.geometry / .crs / .total_bounds）；
6. **认识**三种基础几何类型：Point / LineString / Polygon；
7. **掌握**几何属性：.area / .length / .centroid / .bounds；
8. **掌握** `gdf.plot()` 一行画地图 + 颜色映射；
9. **掌握**从 CSV（含 lon/lat 列）创建 GeoDataFrame；
10. **掌握** `gdf.to_file()` 写入 Shapefile。

---

## 📂 配套数据文件

本节课的目录结构：

```
lesson16/
├── Lesson16_GeoPandas基础.ipynb     ← 你正在看的
└── data/
    ├── provinces_simple.shp        ← 7 个简化省份多边形（含 .shx/.dbf/.prj/.cpg 同名兄弟）
    ├── weather_stations.shp        ← 7 个观测站点位
    └── weather_stations.csv        ← 同 7 站的 CSV 版（用于演示从 CSV 建 GeoDataFrame）
```

**⚠️ 重要**：Shapefile 是一组文件！打开 `data/` 文件夹你会看到每个 .shp 都有 4-5 个**同名兄弟**——必须**全部**齐全才能读取。

---

## 🚀 震撼时刻 —— 一行画地图

**先来体验 GeoPandas 的力量**：一行代码画出中国 7 个省份的地图。

In [ ]:
import geopandas as gpd

provinces = gpd.read_file("data/provinces_simple.shp")
provinces.plot()

**🌟 看到了吗？两行代码就画出了一张地图！**

**这就是 GeoPandas 的力量**——把 Pandas 的优雅扩展到了空间数据领域。

**今天结束**，你将能：
- 读取真实的 Shapefile 数据
- 查看几何信息和属性
- 给地图上色（按属性渲染）
- 把分析结果导出为 Shapefile

---

## 一、GIS 数据基础

### 1.1 矢量 vs 栅格

**地理空间数据有两大类**：

| | 矢量数据（vector） | 栅格数据（raster） |
|---|---|---|
| 表示方式 | 几何对象（点/线/面）| 像素网格 |
| 存储 | 坐标列表 | 二维数组 |
| 例子 | 行政边界、道路、观测站 | 卫星影像、DEM 高程、温度场 |
| 工具 | **GeoPandas**（今天）| Rasterio（第 18 课）|

**今天学的是矢量数据**——下下次课学栅格。

### 1.2 矢量数据的两个核心：几何 + 属性

**每个矢量对象都包含两部分**：

1. **几何（geometry）**：它在地图上的形状和位置
2. **属性（attributes）**：与它关联的数据（名称、人口、面积等）

**例子**：

```
  几何                            属性
  ─────                          ─────
  Point(116.40, 39.90)            station_id=BJ001, city=北京, elevation=43
  Polygon([...])                  name=北京, region=华北, area_km2=16410
```

**🌟 这就是 GeoDataFrame 的本质**：
- 像普通 DataFrame 一样的多列属性
- **多了一个特殊的 `geometry` 列** —— 存储几何对象

### 1.3 CRS —— 坐标参考系

**CRS（Coordinate Reference System）告诉你'坐标值是什么含义'**。

**两类常见 CRS**：

| 类型 | 例子 | 单位 | 适合 |
|---|---|---|---|
| **地理坐标系**（经纬度）| WGS84 / EPSG:4326 | **度** | 全球范围、共享数据 |
| **投影坐标系** | UTM Zone 49N / EPSG:32649 | **米** | 测量距离/面积 |

**关键认知**：
- 同样是'坐标 (116.40, 39.90)'
- 在 EPSG:4326 中表示**经度 116.40°，纬度 39.90°**（北京）
- 在另一个 CRS 中可能毫无意义
- **没有 CRS 的几何数据是'裸数据'，不能在地图上准确定位**

**📌 经验法则**：
- 全球数据/共享数据 → 用 EPSG:4326（WGS84 经纬度）
- 想算距离/面积（米）→ 转投影坐标系（如 EPSG:3857 或 UTM）

### 1.4 Shapefile 的文件结构（高频踩坑）

**Shapefile 不是一个文件！而是一组文件**。

**最常见的 5 个组成部分**（同名不同后缀）：

| 后缀 | 必需？ | 内容 |
|---|---|---|
| **.shp** | ✅ 必需 | 几何数据（核心）|
| **.shx** | ✅ 必需 | 几何索引（加速查询）|
| **.dbf** | ✅ 必需 | 属性表（dBase 格式）|
| **.prj** | ⭐ 强烈推荐 | CRS 信息（没它就没坐标系）|
| **.cpg** | 推荐 | 字符编码 |

**⚠️ 高频踩坑**：
- 拷贝/分享 Shapefile 必须**所有同名文件一起**
- 缺 `.prj` → 不知道 CRS
- 缺 `.dbf` → 没有属性
- 缺 `.shp` 或 `.shx` → 直接报错

In [ ]:
# 看一下 data/ 目录里 Shapefile 的真实组成
import os
for f in sorted(os.listdir("data")):
    print(f)

**看到了吗？**——`provinces_simple` 和 `weather_stations` 各自有 5 个同名兄弟——**这就是 Shapefile 的真面目**。

---

## 二、读取矢量数据 —— gpd.read_file()

**gpd.read_file 是 GeoPandas 的入口**——和 Pandas 的 read_csv 一样简单：

In [ ]:
import geopandas as gpd

# 读取 Shapefile —— 只指定 .shp 文件，会自动找同名兄弟
provinces = gpd.read_file("data/provinces_simple.shp")
provinces

**🌟 关键观察**：

1. 返回的是 **GeoDataFrame**（GeoPandas 的核心结构）
2. **像 Pandas DataFrame 一样**：name、region、area_km2、pop_millio 都是普通列
3. **多了一个 geometry 列**：每行是一个 POLYGON 几何对象
4. 注意 `pop_millio` —— **Shapefile 列名最多 10 字符**，被截断了！这是高频踩坑

In [ ]:
# 读取观测站点（点几何）
stations = gpd.read_file("data/weather_stations.shp")
stations

**这次的 geometry 是 POINT**——点几何。

### 2.1 GeoDataFrame 的特殊属性

**GeoDataFrame 在 Pandas DataFrame 基础上多了 3 个特殊属性**：

In [ ]:
# 1. .geometry —— 几何列（一个 GeoSeries）
print(provinces.geometry.head())
print("\n类型：", type(provinces.geometry))

In [ ]:
# 2. .crs —— 坐标参考系
print("省份 CRS：", provinces.crs)
print("观测站 CRS：", stations.crs)

In [ ]:
# 3. .total_bounds —— 整个数据集的边界框 [xmin, ymin, xmax, ymax]
print("省份的边界框：", provinces.total_bounds)
# 解读：x 范围（经度），y 范围（纬度）

### 2.2 像 Pandas 一样查看数据

**所有 Pandas 的方法 GeoDataFrame 都能用**：

In [ ]:
# 形状
print("形状：", provinces.shape)

# 列名
print("列名：", provinces.columns.tolist())

# 看类型
print("\n类型：")
print(provinces.dtypes)

**📌 注意 dtypes 中的 `geometry` 列类型是 `geometry`**——这是 GeoPandas 特有的类型。

In [ ]:
# 像 Pandas 一样筛选 —— 完全继承！
huabei = provinces[provinces["region"] == "华北"]
huabei

In [ ]:
# 按属性排序 —— 完全继承！
by_pop = provinces.sort_values("pop_millio", ascending=False)
by_pop[["name", "pop_millio"]]

**🌟 你看 —— 第 14 课学的所有 Pandas 技能都能直接用！**

GeoPandas 就是 Pandas + 几何数据处理能力。

---

## 🎯 即时练习①

**练习目标**：熟悉 GeoDataFrame 的基础操作。

**练习题**：

1. 读取 `data/weather_stations.shp` 到变量 `stations`；
2. 打印 stations 的 `shape`、`crs`、`total_bounds`；
3. 查看前 3 行（用 head(3)）；
4. 筛选出 region 为 '华南' 的观测站；
5. 按 elevation（海拔）降序排列；
6. 统计各 region 的观测站数量（用 value_counts，第 14 课的神技）。

In [ ]:
# ===== 即时练习①.1-3 =====
import geopandas as gpd




In [ ]:
# ===== 即时练习①.4 筛选华南 =====




In [ ]:
# ===== 即时练习①.5 按 elevation 降序 =====




In [ ]:
# ===== 即时练习①.6 各 region 站点数 =====




---

## 三、三种基础几何类型

**Shapely** 库定义了 GeoPandas 用的所有几何类型——**今天先认识 3 种最基础的**：

| 类型 | 含义 | 例子 |
|---|---|---|
| **Point** | 点 | 观测站、城市中心、采样点 |
| **LineString** | 线 | 河流、道路、等高线 |
| **Polygon** | 多边形（面）| 行政边界、流域、土地利用区 |

（还有 MultiPoint / MultiLineString / MultiPolygon，今天先了解 3 种基础）

### 3.1 创建几何对象

In [ ]:
from shapely.geometry import Point, LineString, Polygon

# 1. Point —— 一个点（经度, 纬度）
beijing = Point(116.40, 39.90)
print("北京点：", beijing)

# 2. LineString —— 一条线（多个点的列表）
line = LineString([(116.40, 39.90), (121.47, 31.23), (113.27, 23.13)])
print("\n北京-上海-广州连线：", line)

# 3. Polygon —— 一个多边形（首尾要闭合）
poly = Polygon([(116, 39), (118, 39), (118, 41), (116, 41), (116, 39)])
print("\n一个矩形多边形：", poly)

**📌 关键认知**：
- Point：单一点 `(x, y)`
- LineString：点的列表
- Polygon：点的列表（首尾相同，构成闭合环）

**Jupyter 里直接输出几何对象会显示图形预览**——试试看！

In [ ]:
# 直接输出几何对象 —— Jupyter 会显示图形预览
beijing

In [ ]:
line

In [ ]:
poly

---

## 四、几何属性 —— 自动计算的空间信息

**GeoSeries 上有几个非常实用的几何属性**——**自动算出**面积/长度/中心点等：

### 4.1 .area / .length —— 面积和长度

In [ ]:
# 多边形的面积
provinces["calculated_area"] = provinces.geometry.area
provinces[["name", "area_km2", "calculated_area"]]

**⚠️ 注意**：
- `calculated_area` 的单位是 **平方度**（不是平方公里）！
- 因为现在 CRS 是 EPSG:4326（经纬度，单位是度）
- 想得到平方公里 → 必须先**转投影坐标系**（下次课讲）

**📌 这就是 CRS 重要性的真实例子**——**没有正确的 CRS，几何计算的数字毫无意义**。

In [ ]:
# 把 LineString 放进 GeoSeries 看长度
from shapely.geometry import LineString
import geopandas as gpd

lines_gdf = gpd.GeoDataFrame({
    "name": ["北京-上海", "上海-广州"],
    "geometry": [
        LineString([(116.40, 39.90), (121.47, 31.23)]),
        LineString([(121.47, 31.23), (113.27, 23.13)]),
    ]
}, crs="EPSG:4326")

lines_gdf["length_degrees"] = lines_gdf.geometry.length
lines_gdf

### 4.2 .centroid —— 几何中心点

In [ ]:
# 每个省份的中心点
provinces["center"] = provinces.geometry.centroid
provinces[["name", "center"]]

**🌟 .centroid 实战用途**：
- 在地图上给每个省份**贴标签**（标签放中心点）
- 把多边形数据**降维**为点数据
- 计算两个区域之间的'**中心距离**'

### 4.3 .bounds —— 每个几何的边界框

In [ ]:
# 每个几何的最小外包矩形
provinces.geometry.bounds.head()

**📌 .bounds 返回 4 列**：minx, miny, maxx, maxy（每行一个几何的边界）。

### 4.4 几何属性总结

| 属性 | 含义 | 适用 |
|---|---|---|
| `.area` | 面积 | Polygon |
| `.length` | 长度（周长）| LineString / Polygon |
| `.centroid` | 中心点 | 所有 |
| `.bounds` | 边界框 | 所有 |
| `.x` / `.y` | 坐标 | Point only |

**📌 这些都是属性，不是方法 —— 不用括号！**

In [ ]:
# Point 的 .x / .y
stations["lon"] = stations.geometry.x
stations["lat"] = stations.geometry.y
stations[["city", "lon", "lat"]]

---

## 🎯 即时练习②

**练习目标**：综合运用几何属性。

**练习题**：用 `provinces` 完成：

1. 添加新列 `geo_area`，存几何面积（注意：单位是平方度）；
2. 添加新列 `centroid_lon` 和 `centroid_lat`，存每个省份中心点的经纬度（提示：先 .centroid，再 .x / .y）；
3. 找出**'按几何面积最大'的省份**；
4. 计算所有省份**总面积**（所有几何 area 之和）。

In [ ]:
# ===== 即时练习② =====
import geopandas as gpd
provinces = gpd.read_file("data/provinces_simple.shp")

# 任务1：添加 geo_area




In [ ]:
# 任务2：添加 centroid_lon、centroid_lat




In [ ]:
# 任务3：几何面积最大的省份




In [ ]:
# 任务4：所有省份的总面积




---

## 五、画地图 —— gdf.plot()

**GeoPandas 的杀手级功能**——一行画地图。

### 5.1 最简单的 plot()

In [ ]:
import matplotlib.pyplot as plt

# 一行画图
provinces.plot()
plt.show()

### 5.2 设置颜色、大小、边框

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

provinces.plot(
    ax=ax,
    color="lightblue",     # 填充颜色
    edgecolor="black",     # 边框颜色
    linewidth=1            # 边框线宽
)

ax.set_title("7 个省份分布", fontsize=14)
plt.show()

### 5.3 ⭐ 按属性着色（专题地图）

In [ ]:
# 按人口着色 —— 这就是'专题地图'
fig, ax = plt.subplots(figsize=(10, 8))

provinces.plot(
    ax=ax,
    column="pop_millio",   # 按这个列着色
    cmap="OrRd",           # 色卡：橙红色系
    legend=True,           # 显示图例
    edgecolor="black"
)

ax.set_title("7 个省份的人口分布（百万）", fontsize=14)
plt.show()

**🌟 这就是'专题地图'**！按某个属性渲染颜色——一目了然。

**常用色卡（cmap）**：
- 数值连续：`'viridis'`、`'plasma'`、`'OrRd'`、`'Blues'`、`'Greens'`
- 类别离散：`'tab10'`、`'Set1'`、`'Pastel1'`

### 5.4 多层叠加 —— 省份+观测站

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# 第一层：省份（背景）
provinces.plot(
    ax=ax,
    color="lightyellow",
    edgecolor="gray",
    linewidth=0.5
)

# 第二层：观测站（叠加在省份之上）—— 关键：用同一个 ax
stations.plot(
    ax=ax,
    color="red",
    markersize=80,
    marker="^"             # 三角形标记
)

ax.set_title("省份 + 观测站位置", fontsize=14)
plt.show()

**🌟 多层叠加的关键 —— 用同一个 `ax` 参数**！

这就是制作真实地图的方式：
- 第一层：背景（省份/国家）
- 第二层：要素（观测站/河流）
- 第三层：标注/图例

---

## 六、从 CSV 创建 GeoDataFrame —— 实用神技

**真实工作中**，你拿到的常常是带 lon/lat 列的 CSV——**不是 Shapefile**。

**怎么把它变成 GeoDataFrame？**

In [ ]:
import pandas as pd
import geopandas as gpd

# 第 1 步：用 Pandas 读 CSV
df = pd.read_csv("data/weather_stations.csv")
df.head()

In [ ]:
# 第 2 步：用 gpd.points_from_xy 把 lon/lat 列变成几何
geometry = gpd.points_from_xy(df["longitude"], df["latitude"])

# 第 3 步：构造 GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")

gdf.head()

**🔑 三步法**：

1. **`pd.read_csv()`** —— 读 CSV 成普通 DataFrame
2. **`gpd.points_from_xy(lon, lat)`** —— 把 lon/lat 列变成 Point 几何
3. **`gpd.GeoDataFrame(df, geometry=..., crs=...)`** —— 包装成 GeoDataFrame

**⚠️ 关键 —— `crs="EPSG:4326"` 必须指定**！否则不知道 lon/lat 是什么坐标系。

In [ ]:
# 现在 gdf 是真正的 GeoDataFrame —— 可以画地图！
fig, ax = plt.subplots(figsize=(10, 8))
gdf.plot(
    ax=ax,
    column="annual_rainfall",
    cmap="Blues",
    markersize=100,
    legend=True
)
ax.set_title("观测站的年降水量", fontsize=14)
plt.show()

---

## 七、写入文件 —— gdf.to_file()

**和 Pandas 的 to_csv 类似**——一行写入 Shapefile：

In [ ]:
# 把刚才从 CSV 创建的 GeoDataFrame 写成 Shapefile
gdf.to_file("output_stations.shp", encoding="utf-8")

# 验证 —— 看一下生成的文件
import os
for f in sorted(os.listdir(".")):
    if f.startswith("output_stations"):
        print(f, os.path.getsize(f), "字节")

**📌 注意**：`to_file` 会**自动生成 5 个文件**（.shp/.shx/.dbf/.prj/.cpg）—— 这就是 Shapefile 的真相。

### 7.1 更现代的格式：GeoPackage 和 GeoJSON

**Shapefile 有几个老问题**：
- 列名 ≤10 字符
- 一组 5 个文件（容易丢失）
- 不支持现代日期时间类型

**现代替代品**：

| 格式 | 后缀 | 特点 |
|---|---|---|
| **GeoPackage** | `.gpkg` | 单文件、SQLite 内核、列名无限制（推荐！）|
| **GeoJSON** | `.geojson` | 文本格式、Web 友好、易交换 |

In [ ]:
# GeoPackage（推荐！）
gdf.to_file("output_stations.gpkg", driver="GPKG", layer="stations")

# GeoJSON
gdf.to_file("output_stations.geojson", driver="GeoJSON")

for f in sorted(os.listdir(".")):
    if f.startswith("output_stations.") and not f.endswith(("shp", "shx", "dbf", "prj", "cpg")):
        print(f, os.path.getsize(f), "字节")

**📌 经验法则**：
- **存档/分享 → GeoPackage**（单文件、属性无限制）
- **Web 应用/JS 库 → GeoJSON**
- **兼容老 GIS 软件 → Shapefile**（仍是事实标准）

---

## 🎯 即时练习③（综合实战）

**练习目标**：完成'读 CSV → 创建 GeoDataFrame → 画地图 → 导出'完整工作流。

**任务**：

**步骤1**：用 `pd.read_csv` 读取 `data/weather_stations.csv`；

**步骤2**：用 `gpd.points_from_xy` 创建几何，构造 GeoDataFrame（CRS 设为 EPSG:4326）；

**步骤3**：添加新列 `is_humid`：`True` 表示 annual_rainfall > 1000，否则 False；

**步骤4**：画一张图：背景是省份（淡灰色），前景是观测站（按 elevation 着色）；

**步骤5**：把 GeoDataFrame 导出为 `output_humid_stations.gpkg`（GeoPackage 格式）；

**步骤6**：单独把 `is_humid==True` 的子集导出为 `output_humid_only.geojson`。

**对比第 11 课的工作流** —— 现在多了'空间维度'，但代码同样优雅。

In [ ]:
# ===== 即时练习③ 综合实战 =====
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# 步骤1：读 CSV




In [ ]:
# 步骤2：构造 GeoDataFrame




In [ ]:
# 步骤3：添加 is_humid 列




In [ ]:
# 步骤4：双层地图（省份背景 + 站点按海拔着色）




In [ ]:
# 步骤5：写入 GeoPackage




In [ ]:
# 步骤6：仅 is_humid=True 的子集写入 GeoJSON




---

## 八、本节课小结

### 知识点回顾

| 知识点 | 关键语法 |
|---|---|
| import 约定 | `import geopandas as gpd` |
| 读取矢量 | `gpd.read_file('file.shp')` |
| 写入文件 | `gdf.to_file('out.shp')` 或 `out.gpkg` |
| 几何列 | `gdf.geometry`（GeoSeries）|
| 坐标系 | `gdf.crs` |
| 边界框 | `gdf.total_bounds` |
| 几何属性 | `.area / .length / .centroid / .bounds` |
| Point 坐标 | `point.x / point.y` |
| 创建几何 | `Point(x,y)` / `LineString([...])` / `Polygon([...])` |
| 从 CSV 建 GeoDataFrame | `gpd.points_from_xy(lon, lat)` |
| 画地图 | `gdf.plot()` |
| 专题地图 | `.plot(column='col', cmap='OrRd', legend=True)` |
| 多层叠加 | 共享 `ax` 参数 |

### 重点强调（重要程度从高到低）

1. ⭐⭐⭐ **`gpd.read_file()`** —— GIS 数据的入口
2. ⭐⭐⭐ **GeoDataFrame = DataFrame + geometry 列**
3. ⭐⭐⭐ **CRS 必须指定** —— 没 CRS 的数据是裸数据
4. ⭐⭐ **几何属性是属性不是方法**（`.area` 不是 `.area()`）
5. ⭐⭐ **`gdf.plot()` 画地图** —— 一行专题地图
6. ⭐⭐ **`gpd.points_from_xy()`** —— CSV→GeoDataFrame 的关键
7. ⭐ **Shapefile 是一组文件** —— 不是单个文件
8. ⭐ **Shapefile 列名 ≤10 字符** —— 用 GeoPackage 没此限制

### 课后作业

**1. 【基础】GeoDataFrame 基础操作（必做）**

用 `data/provinces_simple.shp`：
1. 读取并打印 shape、crs、total_bounds；
2. 筛选出 `region == '华东'` 的省份；
3. 添加新列 `geo_area`（几何面积）和 `geo_length`（周长）；
4. 找出'人口最多的省份'。

**2. 【进阶】专题地图（必做）**

用 `data/provinces_simple.shp`：
1. 画一张'按 region 着色'的地图（cmap='Set2'）；
2. 画一张'按 area_km2 着色'的地图（cmap='Greens'）；
3. 画一张'省份 + 站点中心标签'的地图（在每个省份的中心点画个 Point 标记）；
4. 都加标题。

**3. 【挑战】完整 ETL 流水线（必做）**

1. 读 `data/weather_stations.csv` → DataFrame；
2. 创建 GeoDataFrame（带 CRS）；
3. 添加新列 `climate_label`：rainfall > 1500 标 'humid'，500-1500 标 'mild'，< 500 标 'dry'（用 np.where 嵌套）；
4. 画一张地图：背景省份 + 站点按 climate_label 着色（cmap='Set1'）；
5. 导出为 `output_climate_stations.gpkg`（GeoPackage）。

---

下节课预告：**第17次课 GeoPandas 空间分析** —— 缓冲区分析（buffer）、空间查询（contains/within/intersects）、空间连接（sjoin）—— 真正的'GIS 空间分析'！

---
---

# 📎 附录：参考答案

> **建议先独立完成所有练习题再查看答案。**

In [ ]:
# ===== 即时练习① 参考答案 =====
import geopandas as gpd

# 1-3 读取并查看
stations = gpd.read_file("data/weather_stations.shp")
print("shape:", stations.shape)
print("crs:", stations.crs)
print("total_bounds:", stations.total_bounds)
print("\n前 3 行：")
print(stations.head(3))

In [ ]:
# 4. 筛选华南
huanan = stations[stations["region"] == "华南"]
print(huanan)

In [ ]:
# 5. 按 elevation 降序
by_elev = stations.sort_values("elevation", ascending=False)
print(by_elev[["city", "elevation"]])

In [ ]:
# 6. 各 region 站点数
print(stations["region"].value_counts())

In [ ]:
# ===== 即时练习② 参考答案 =====
import geopandas as gpd
provinces = gpd.read_file("data/provinces_simple.shp")

# 1. geo_area
provinces["geo_area"] = provinces.geometry.area

# 2. centroid_lon, centroid_lat
centers = provinces.geometry.centroid
provinces["centroid_lon"] = centers.x
provinces["centroid_lat"] = centers.y

print(provinces[["name", "geo_area", "centroid_lon", "centroid_lat"]])

In [ ]:
# 3. 几何面积最大
largest = provinces.sort_values("geo_area", ascending=False).head(1)
print("几何面积最大的省份：")
print(largest[["name", "geo_area"]])

In [ ]:
# 4. 总面积
total = provinces.geometry.area.sum()
print(f"所有省份的总几何面积：{total:.2f} 平方度")

In [ ]:
# ===== 即时练习③ 综合实战 参考答案 =====
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

# 步骤1：读 CSV
df = pd.read_csv("data/weather_stations.csv")
print(df.head())

# 步骤2：构造 GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

# 步骤3：添加 is_humid
gdf["is_humid"] = gdf["annual_rainfall"] > 1000
print("\n添加 is_humid 后：")
print(gdf[["city", "annual_rainfall", "is_humid"]])

In [ ]:
# 步骤4：双层地图
provinces = gpd.read_file("data/provinces_simple.shp")

fig, ax = plt.subplots(figsize=(10, 8))

# 第一层：省份背景
provinces.plot(
    ax=ax,
    color="lightgray",
    edgecolor="darkgray",
    linewidth=0.8
)

# 第二层：站点按海拔着色
gdf.plot(
    ax=ax,
    column="elevation",
    cmap="viridis",
    markersize=120,
    edgecolor="black",
    legend=True
)

ax.set_title("观测站海拔分布", fontsize=14)
plt.show()

In [ ]:
# 步骤5：导出 GeoPackage
gdf.to_file("output_humid_stations.gpkg", driver="GPKG", layer="stations")
print("output_humid_stations.gpkg 已写入！")

In [ ]:
# 步骤6：仅湿润子集导出 GeoJSON
humid_only = gdf[gdf["is_humid"]]
humid_only.to_file("output_humid_only.geojson", driver="GeoJSON")
print(f"output_humid_only.geojson 已写入！（共 {len(humid_only)} 站）")

---

## 课后作业参考答案

In [ ]:
# ===== 课后作业 1 参考答案 =====
import geopandas as gpd

provinces = gpd.read_file("data/provinces_simple.shp")

# 1
print("shape:", provinces.shape)
print("crs:", provinces.crs)
print("total_bounds:", provinces.total_bounds)

# 2 华东
east = provinces[provinces["region"] == "华东"]
print("\n=== 华东省份 ===")
print(east[["name", "region"]])

# 3 添加 geo_area / geo_length
provinces["geo_area"] = provinces.geometry.area
provinces["geo_length"] = provinces.geometry.length

# 4 人口最多的省份
most_pop = provinces.sort_values("pop_millio", ascending=False).head(1)
print("\n=== 人口最多 ===")
print(most_pop[["name", "pop_millio"]])

In [ ]:
# ===== 课后作业 2 参考答案 =====
import geopandas as gpd
import matplotlib.pyplot as plt

provinces = gpd.read_file("data/provinces_simple.shp")

# 1. 按 region 着色
fig, ax = plt.subplots(figsize=(10, 8))
provinces.plot(ax=ax, column="region", cmap="Set2", edgecolor="black", legend=True)
ax.set_title("按区域着色", fontsize=14)
plt.show()

# 2. 按 area_km2 着色
fig, ax = plt.subplots(figsize=(10, 8))
provinces.plot(ax=ax, column="area_km2", cmap="Greens", edgecolor="black", legend=True)
ax.set_title("按面积着色（km²）", fontsize=14)
plt.show()

# 3. 省份 + 中心点标签
fig, ax = plt.subplots(figsize=(10, 8))
provinces.plot(ax=ax, color="lightyellow", edgecolor="gray")
centers = provinces.geometry.centroid
centers.plot(ax=ax, color="red", markersize=80, marker="*")
ax.set_title("省份 + 中心点", fontsize=14)
plt.show()

In [ ]:
# ===== 课后作业 3 参考答案 =====
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt

# 1. 读 CSV
df = pd.read_csv("data/weather_stations.csv")

# 2. 构造 GeoDataFrame
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

# 3. climate_label
gdf["climate_label"] = np.where(
    gdf["annual_rainfall"] > 1500, "humid",
    np.where(gdf["annual_rainfall"] >= 500, "mild", "dry")
)

# 4. 地图：省份 + 站点按 climate_label 着色
provinces = gpd.read_file("data/provinces_simple.shp")
fig, ax = plt.subplots(figsize=(10, 8))
provinces.plot(ax=ax, color="lightyellow", edgecolor="gray")
gdf.plot(ax=ax, column="climate_label", cmap="Set1", markersize=120, legend=True, edgecolor="black")
ax.set_title("观测站气候类型分布", fontsize=14)
plt.show()

# 5. 导出 GeoPackage
gdf.to_file("output_climate_stations.gpkg", driver="GPKG", layer="stations")
print("output_climate_stations.gpkg 已写入！")
print(gdf[["city", "annual_rainfall", "climate_label"]])